In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_DIR = Path("..").resolve()

RAW_DATA = PROJECT_DIR / "data" / "raw" / "ecommerce_sales.csv"
PROCESSED_DATA = PROJECT_DIR / "data" / "processed" / "sales_clean.csv"

RAW_DATA

WindowsPath('D:/Self Project/SumoPod API/Profile/projects/sales_dashboard/data/raw/ecommerce_sales.csv')

In [3]:
df = pd.read_csv(RAW_DATA)
df.head()

,Order_ID,Order_Date,Customer_Name,Customer_Segment,Country,Region,Product_Category,Product_Name,Quantity,Unit_Price,Discount_Percent,Total_Sales,Shipping_Cost,Profit,Payment_Method
0,ORD-11121,2023-01-02,Karen Suzuki,Corporate,United States,North America,Technology,Wireless Bluetooth Headphones,3,99.43,0,298.29,9.31,124.92,Cash on Delivery
1,ORD-11244,2023-01-02,John Johansson,Corporate,Spain,Europe,Technology,Mechanical Gaming Keyboard,4,97.93,20,313.38,14.31,83.62,Cash on Delivery
2,ORD-10325,2023-01-03,Jessica Garcia,Consumer,Mexico,North America,Office Supplies,Binder Clips Assorted 48pc,2,10.74,0,21.48,8.12,3.69,Credit Card
3,ORD-10467,2023-01-03,Clara Taylor,Corporate,Italy,Europe,Technology,Webcam HD 1080p,2,61.86,15,105.16,10.79,26.32,PayPal
4,ORD-11454,2023-01-05,Felix Thomas,Consumer,Italy,Europe,Furniture,Standing Desk Converter,4,330.67,20,1058.14,11.09,253.44,Credit Card


In [4]:
print(df.head(5))
print(df.columns)
print(df.info())


    Order_ID  Order_Date   Customer_Name Customer_Segment        Country  \
0  ORD-11121  2023-01-02    Karen Suzuki        Corporate  United States   
1  ORD-11244  2023-01-02  John Johansson        Corporate          Spain   
2  ORD-10325  2023-01-03  Jessica Garcia         Consumer         Mexico   
3  ORD-10467  2023-01-03    Clara Taylor        Corporate          Italy   
4  ORD-11454  2023-01-05    Felix Thomas         Consumer          Italy   

          Region Product_Category                   Product_Name  Quantity  \
0  North America       Technology  Wireless Bluetooth Headphones         3   
1         Europe       Technology     Mechanical Gaming Keyboard         4   
2  North America  Office Supplies     Binder Clips Assorted 48pc         2   
3         Europe       Technology                Webcam HD 1080p         2   
4         Europe        Furniture        Standing Desk Converter         4   

   Unit_Price  Discount_Percent  Total_Sales  Shipping_Cost  Profit  \
0  

In [5]:
sales = df.copy()

In [6]:
sales.columns = (
    sales.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

sales.columns

Index(['order_id', 'order_date', 'customer_name', 'customer_segment',
       'country', 'region', 'product_category', 'product_name', 'quantity',
       'unit_price', 'discount_percent', 'total_sales', 'shipping_cost',
       'profit', 'payment_method'],
      dtype='object')

In [7]:
sales.head()

,order_id,order_date,customer_name,customer_segment,country,region,product_category,product_name,quantity,unit_price,discount_percent,total_sales,shipping_cost,profit,payment_method
0,ORD-11121,2023-01-02,Karen Suzuki,Corporate,United States,North America,Technology,Wireless Bluetooth Headphones,3,99.43,0,298.29,9.31,124.92,Cash on Delivery
1,ORD-11244,2023-01-02,John Johansson,Corporate,Spain,Europe,Technology,Mechanical Gaming Keyboard,4,97.93,20,313.38,14.31,83.62,Cash on Delivery
2,ORD-10325,2023-01-03,Jessica Garcia,Consumer,Mexico,North America,Office Supplies,Binder Clips Assorted 48pc,2,10.74,0,21.48,8.12,3.69,Credit Card
3,ORD-10467,2023-01-03,Clara Taylor,Corporate,Italy,Europe,Technology,Webcam HD 1080p,2,61.86,15,105.16,10.79,26.32,PayPal
4,ORD-11454,2023-01-05,Felix Thomas,Consumer,Italy,Europe,Furniture,Standing Desk Converter,4,330.67,20,1058.14,11.09,253.44,Credit Card


In [8]:
# Convert Dates
sales["order_date"] = pd.to_datetime(sales["order_date"])

sales["order_date"].min(), sales["order_date"].max()

(Timestamp('2023-01-02 00:00:00'), Timestamp('2025-12-31 00:00:00'))

In [9]:
# Feature engineering
sales["order_year"] = sales["order_date"].dt.year
sales["order_month"] = sales["order_date"].dt.month
sales["order_month_name"] = sales["order_date"].dt.strftime("%b")
sales["order_year_month"] = sales["order_date"].dt.to_period("M").astype(str)
sales["order_quarter"] = sales["order_date"].dt.to_period("Q").astype(str)

sales["gross_sales"] = sales["quantity"] * sales["unit_price"]
sales["discount_amount"] = sales["gross_sales"] * (sales["discount_percent"] / 100)
sales["net_sales_check"] = sales["gross_sales"] - sales["discount_amount"]

sales["profit_margin"] = sales["profit"] / sales["total_sales"]
sales["shipping_cost_ratio"] = sales["shipping_cost"] / sales["total_sales"]

sales.head()

,order_id,order_date,customer_name,customer_segment,country,region,product_category,product_name,quantity,unit_price,...,order_year,order_month,order_month_name,order_year_month,order_quarter,gross_sales,discount_amount,net_sales_check,profit_margin,shipping_cost_ratio
0,ORD-11121,2023-01-02,Karen Suzuki,Corporate,United States,North America,Technology,Wireless Bluetooth Headphones,3,99.43,...,2023,1,Jan,2023-01,2023Q1,298.29,0.000,298.290,0.418787,0.031211
1,ORD-11244,2023-01-02,John Johansson,Corporate,Spain,Europe,Technology,Mechanical Gaming Keyboard,4,97.93,...,2023,1,Jan,2023-01,2023Q1,391.72,78.344,313.376,0.266833,0.045663
2,ORD-10325,2023-01-03,Jessica Garcia,Consumer,Mexico,North America,Office Supplies,Binder Clips Assorted 48pc,2,10.74,...,2023,1,Jan,2023-01,2023Q1,21.48,0.000,21.480,0.171788,0.378026
3,ORD-10467,2023-01-03,Clara Taylor,Corporate,Italy,Europe,Technology,Webcam HD 1080p,2,61.86,...,2023,1,Jan,2023-01,2023Q1,123.72,18.558,105.162,0.250285,0.102606
4,ORD-11454,2023-01-05,Felix Thomas,Consumer,Italy,Europe,Furniture,Standing Desk Converter,4,330.67,...,2023,1,Jan,2023-01,2023Q1,1322.68,264.536,1058.144,0.239515,0.010481


In [10]:
# Total Sales
sales[["gross_sales", "discount_amount", "net_sales_check", "total_sales"]].head()

,gross_sales,discount_amount,net_sales_check,total_sales
0,298.29,0.000,298.290,298.29
1,391.72,78.344,313.376,313.38
2,21.48,0.000,21.480,21.48
3,123.72,18.558,105.162,105.16
4,1322.68,264.536,1058.144,1058.14


In [11]:
# Selisih Sales
sales["sales_diff"] = sales["total_sales"] - sales["net_sales_check"]

sales["sales_diff"].describe()

count    2000.000000
mean       -0.000095
std         0.002485
min        -0.005000
25%        -0.002000
50%         0.000000
75%         0.002000
max         0.005000
Name: sales_diff, dtype: float64

In [12]:
total_sales = sales["total_sales"].sum()
total_profit = sales["profit"].sum()
total_orders = sales["order_id"].nunique()
total_customers = sales["customer_name"].nunique()
avg_order_value = total_sales / total_orders
profit_margin = total_profit / total_sales

{
    "total_sales": total_sales,
    "total_profit": total_profit,
    "total_orders": total_orders,
    "total_customers": total_customers,
    "avg_order_value": avg_order_value,
    "profit_margin": profit_margin,
}

{'total_sales': np.float64(484559.33999999997),
 'total_profit': np.float64(158872.32),
 'total_orders': 2000,
 'total_customers': 1534,
 'avg_order_value': np.float64(242.27966999999998),
 'profit_margin': np.float64(0.32786968877743644)}

In [13]:
monthly_sales = (
    sales.groupby("order_year_month", as_index=False)
    .agg(
        total_sales=("total_sales", "sum"),
        total_profit=("profit", "sum"),
        total_orders=("order_id", "nunique"),
    )
)

monthly_sales.head()

,order_year_month,total_sales,total_profit,total_orders
0,2023-01,11275.20,3573.01,47
1,2023-02,16943.67,5980.06,52
2,2023-03,16056.45,5718.77,51
3,2023-04,12185.00,3994.25,48
4,2023-05,8218.47,2591.86,40


In [14]:
category_sales = (
    sales.groupby("product_category", as_index=False)
    .agg(
        total_sales=("total_sales", "sum"),
        total_profit=("profit", "sum"),
        total_orders=("order_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)

category_sales

,product_category,total_sales,total_profit,total_orders,quantity_sold
1,Furniture,256274.68,81171.57,507,1806
3,Technology,139518.22,48268.65,567,2017
0,Clothing & Accessories,69375.63,26112.94,413,1551
2,Office Supplies,19390.81,3319.16,513,1741


In [15]:
region_sales = (
    sales.groupby("region", as_index=False)
    .agg(
        total_sales=("total_sales", "sum"),
        total_profit=("profit", "sum"),
        total_orders=("order_id", "nunique"),
    )
    .sort_values("total_sales", ascending=False)
)

region_sales

,region,total_sales,total_profit,total_orders
1,Europe,137006.20,45672.16,503
3,North America,133876.38,45250.09,578
0,Asia Pacific,121707.51,39116.61,520
4,South America,46051.13,14680.98,191
2,Middle East & Africa,45918.12,14152.48,208


In [16]:
PROCESSED_DATA.parent.mkdir(parents=True, exist_ok=True)

sales.to_csv(PROCESSED_DATA, index=False)

PROCESSED_DATA

WindowsPath('D:/Self Project/SumoPod API/Profile/projects/sales_dashboard/data/processed/sales_clean.csv')

In [17]:
sales.head()
sales.info()
monthly_sales.head()
category_sales
region_sales

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             2000 non-null   object        
 1   order_date           2000 non-null   datetime64[ns]
 2   customer_name        2000 non-null   object        
 3   customer_segment     2000 non-null   object        
 4   country              2000 non-null   object        
 5   region               2000 non-null   object        
 6   product_category     2000 non-null   object        
 7   product_name         2000 non-null   object        
 8   quantity             2000 non-null   int64         
 9   unit_price           2000 non-null   float64       
 10  discount_percent     2000 non-null   int64         
 11  total_sales          2000 non-null   float64       
 12  shipping_cost        2000 non-null   float64       
 13  profit               2000 non-nul

,region,total_sales,total_profit,total_orders
1,Europe,137006.20,45672.16,503
3,North America,133876.38,45250.09,578
0,Asia Pacific,121707.51,39116.61,520
4,South America,46051.13,14680.98,191
2,Middle East & Africa,45918.12,14152.48,208


### EDA


In [18]:
import plotly.express as px

In [19]:
fig = px.line(
    monthly_sales,
    x="order_year_month",
    y="total_sales",
    markers=True,
    title="Monthly Sales Trend",
)

fig.show()

In [20]:
fig = px.line(
    monthly_sales,
    x="order_year_month",
    y="total_profit",
    markers=True,
    title="Monthly Profit Trend",
)

fig.show()

In [21]:
fig = px.bar(
    category_sales,
    x="product_category",
    y="total_sales",
    title="Sales by Product Category",
    text_auto=".2s",
)

fig.show()

In [ ]:
fig = px.bar(
    region_sales,
    x="region",
    y="total_profit",
    title="Profit by Region",
    text_auto=".2s",
)

fig.show()

In [23]:
segment_sales = (
    sales.groupby("customer_segment", as_index=False)
    .agg(
        total_sales=("total_sales", "sum"),
        total_profit=("profit", "sum"),
        total_orders=("order_id", "nunique"),
    )
    .sort_values("total_sales", ascending=False)
)

fig = px.bar(
    segment_sales,
    x="customer_segment",
    y="total_sales",
    color="total_profit",
    title="Sales by Customer Segment",
    text_auto=".2s",
)

fig.show()

In [24]:
top_products = (
    sales.groupby("product_name", as_index=False)
    .agg(
        total_sales=("total_sales", "sum"),
        total_profit=("profit", "sum"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("total_sales", ascending=False)
    .head(10)
)

fig = px.bar(
    top_products,
    x="total_sales",
    y="product_name",
    orientation="h",
    title="Top 10 Products by Sales",
    text_auto=".2s",
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [25]:
category_margin = category_sales.copy()
category_margin["profit_margin"] = (
    category_margin["total_profit"] / category_margin["total_sales"]
)

fig = px.bar(
    category_margin,
    x="product_category",
    y="profit_margin",
    title="Profit Margin by Product Category",
    text_auto=".1%",
)

fig.show()

In [26]:
fig = px.scatter(
    sales,
    x="discount_percent",
    y="profit",
    color="product_category",
    size="total_sales",
    hover_data=["product_name", "country", "customer_segment"],
    title="Discount vs Profit",
)

fig.show()

# Business Insights

Based on the exploratory analysis, several business insights stand out:

1. Total sales reached **484,559.34** with total profit of **158,872.32**, producing an overall profit margin of **32.79%**.
2. **Furniture** is the strongest product category by sales, contributing **256,274.68**, followed by **Technology** with **139,518.22**.
3. **Europe** is the top region by sales with **137,006.20**, closely followed by **North America** with **133,876.38**.
4. The **Consumer** segment is the largest customer segment, generating **256,287.74** in sales from **1,006** orders.
5. Higher discount levels appear to reduce total profit. Orders with **0% discount** generated **50,705.28** profit, while **30% discount** generated only **728.32** profit.

These findings suggest the dashboard should prioritize category performance, regional comparison, customer segment contribution, and discount impact on profitability.


## How Sales, Region, and Discount Strategy Shape E-Commerce Profitability

This analysis explores 2,000 global e-commerce transactions from 2023 to 2025 to understand what drives sales and profit performance. The goal is to identify which product categories, regions, and customer segments contribute the most to revenue, and how discount strategy affects profitability.

## 1. Business Overview

The business generated 484,559.34 in sales and 158,872.32 in profit, with an overall profit margin of 32.79%. This indicates that the company is profitable overall, but profitability may vary across product categories, regions, and discount levels.

## 2. Revenue Drivers

Furniture is the dominant revenue driver, generating 256,274.68 in sales. Technology follows with 139,518.22. This suggests that higher-ticket products contribute more strongly to total sales.

## 3. Regional Performance

Europe leads regional sales with 137,006.20, closely followed by North America at 133,876.38 and Asia Pacific at 121,707.51. The top three regions are relatively balanced, meaning growth strategy should not focus on only one market.

## 4. Customer Segment Contribution

The Consumer segment contributes the largest sales volume, generating 256,287.74 from 1,006 orders. Corporate and Home Office segments are smaller, but still important for profitability and targeted campaigns.

## 5. Discount Impact

Orders without discount generated the highest total profit at 50,705.28. As discount levels increased, total profit declined sharply. Orders with 30% discount generated only 728.32 in profit. This suggests that aggressive discounting may reduce profitability instead of improving business performance.

## 6. Business Recommendation

The company should prioritize Furniture and Technology products, maintain strong regional coverage across Europe, North America, and Asia Pacific, and evaluate discount policies more carefully. Instead of applying high discounts broadly, discounts should be targeted to specific products, segments, or regions where they can increase volume without damaging profit margin.